## 2026 EY AI &amp; Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog.

**Caution**... This notebook requires significant execution time as there are 9,319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer with a typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and the use of cloud computing services.


### Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process. 

In [ ]:
# STRICT Snowflake: install deps if requirements.txt is present in common locations
!pip install -q uv

import os
req_candidates = ['requirements.txt','./requirements.txt','../requirements.txt']
req_path = next((p for p in req_candidates if os.path.exists(p)), None)
if req_path:
    !uv pip install -q -r "{req_path}"
else:
    print('requirements.txt not found; skipping requirements install')


In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

In [ ]:
# ============================================================
# STRICT helpers (Snowflake notebook friendly)
# ============================================================
from pathlib import Path

def read_csv_strict(filename: str) -> pd.DataFrame:
    candidates = [Path(filename), Path('./')/filename, Path('../')/filename]
    for p in candidates:
        try:
            if p.exists():
                return pd.read_csv(p.as_posix())
        except Exception:
            pass
    return pd.read_csv(filename)

# Global STAC catalog (reuse across calls to reduce overhead)
CATALOG = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=pc.sign_inplace
)


### Extracting Landsat Data Using API Calls

The API-based method allows us to efficiently access **Landsat** data for specific coordinates and time periods, ensuring scalability and reproducibility of the process.

Through the API, we can query individual bands or compute indices like **NDMI** on the fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis.

The **compute_Landsat_values** function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:

- A bounding box (bbox) is created around the latitude and longitude coordinates.
- The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.
- The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (**green**, **nir08**, **swir16**, **swir22**) are loaded.
- Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.

**Why the buffer value is 0.00089831**

We want a ~100 m buffer around each point.  
At the equator, 1 degree ≈ 110 km.

Therefore, the degree equivalent of 100 m is:

*buffer_deg ≈ 100 m / 110,000 m per degree ≈ 0.00089831*

This value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.


In [2]:
# Setup
tqdm.pandas()

# ============================================================
# XGBoost-friendly Landsat feature extraction
# - Reuses global STAC client
# - Searches around sample date (±WINDOW_DAYS)
# - Extracts multiple bands + robust indices
# ============================================================
EPS = 1e-10
BBOX_SIZE_DEG = 0.00089831  # ~100m
CLOUD_LT = 20  # relax slightly to reduce missing scenes
WINDOW_DAYS = 45

BANDS = ['blue','green','red','nir08','swir16','swir22']

def _safe_median(x):
    try:
        v = float(x.median(skipna=True).values)
        return np.nan if v == 0 else v
    except Exception:
        return np.nan

def _pick_closest_item(items, sample_dt_utc):
    # items: ItemCollection
    items_sorted = sorted(
        items,
        key=lambda it: abs(pd.to_datetime(it.properties['datetime']).tz_convert('UTC') - sample_dt_utc)
    )
    return items_sorted[0] if items_sorted else None

def compute_Landsat_values(row):
    lat = float(row['Latitude'])
    lon = float(row['Longitude'])
    sample_dt = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    bbox = [
        lon - BBOX_SIZE_DEG/2,
        lat - BBOX_SIZE_DEG/2,
        lon + BBOX_SIZE_DEG/2,
        lat + BBOX_SIZE_DEG/2
    ]

    # Build datetime range around the sample date when possible
    if pd.isna(sample_dt):
        dt_range = '2011-01-01/2015-12-31'
        sample_dt_utc = pd.Timestamp('2013-01-01', tz='UTC')
    else:
        sample_dt_utc = sample_dt.tz_localize('UTC') if sample_dt.tzinfo is None else sample_dt.tz_convert('UTC')
        start = (sample_dt_utc - pd.Timedelta(days=WINDOW_DAYS)).date().isoformat()
        end   = (sample_dt_utc + pd.Timedelta(days=WINDOW_DAYS)).date().isoformat()
        dt_range = f'{start}/{end}'

    try:
        search = CATALOG.search(
            collections=['landsat-c2-l2'],
            bbox=bbox,
            datetime=dt_range,
            query={'eo:cloud_cover': {'lt': CLOUD_LT}},
            limit=50
        )
        items = search.item_collection()
        if not items:
            # fallback to wider range if nothing found
            search = CATALOG.search(
                collections=['landsat-c2-l2'],
                bbox=bbox,
                datetime='2011-01-01/2015-12-31',
                query={'eo:cloud_cover': {'lt': CLOUD_LT}},
                limit=50
            )
            items = search.item_collection()

        if not items:
            return pd.Series({
                'blue': np.nan, 'green': np.nan, 'red': np.nan, 'nir': np.nan, 'swir16': np.nan, 'swir22': np.nan,
                'NDVI': np.nan, 'NDMI': np.nan, 'MNDWI': np.nan, 'NDWI': np.nan, 'NBR': np.nan
            })

        item = _pick_closest_item(items, sample_dt_utc)
        if item is None:
            return pd.Series({
                'blue': np.nan, 'green': np.nan, 'red': np.nan, 'nir': np.nan, 'swir16': np.nan, 'swir22': np.nan,
                'NDVI': np.nan, 'NDMI': np.nan, 'MNDWI': np.nan, 'NDWI': np.nan, 'NBR': np.nan
            })

        signed_item = pc.sign(item)
        data = stac_load([signed_item], bands=BANDS, bbox=bbox).isel(time=0)

        blue = _safe_median(data['blue'].astype('float'))
        green = _safe_median(data['green'].astype('float'))
        red = _safe_median(data['red'].astype('float'))
        nir = _safe_median(data['nir08'].astype('float'))
        swir16 = _safe_median(data['swir16'].astype('float'))
        swir22 = _safe_median(data['swir22'].astype('float'))

        # Indices (robust eps to avoid div0)
        ndvi = (nir - red) / (nir + red + EPS)
        ndmi = (nir - swir16) / (nir + swir16 + EPS)
        mndwi = (green - swir16) / (green + swir16 + EPS)
        ndwi = (green - nir) / (green + nir + EPS)
        nbr = (nir - swir22) / (nir + swir22 + EPS)

        return pd.Series({
            'blue': blue, 'green': green, 'red': red, 'nir': nir, 'swir16': swir16, 'swir22': swir22,
            'NDVI': float(ndvi) if np.isfinite(ndvi) else np.nan,
            'NDMI': float(ndmi) if np.isfinite(ndmi) else np.nan,
            'MNDWI': float(mndwi) if np.isfinite(mndwi) else np.nan,
            'NDWI': float(ndwi) if np.isfinite(ndwi) else np.nan,
            'NBR': float(nbr) if np.isfinite(nbr) else np.nan
        })

    except Exception:
        return pd.Series({
            'blue': np.nan, 'green': np.nan, 'red': np.nan, 'nir': np.nan, 'swir16': np.nan, 'swir22': np.nan,
            'NDVI': np.nan, 'NDMI': np.nan, 'MNDWI': np.nan, 'NDWI': np.nan, 'NBR': np.nan
        })


### Extracting features for the training dataset

### XGBoost-friendly Landsat features

This notebook outputs **raw bands** plus **derived indices** commonly useful for tree-based models like XGBoost: NDVI, NDMI, MNDWI, NDWI, and NBR.

Latitude/Longitude are included only as join keys (do **not** use them directly as model features).

In [3]:
Water_Quality_df = read_csv_strict('water_quality_training_dataset.csv')
display(Water_Quality_df.head())

In [4]:
Water_Quality_df.shape

In [5]:
# Select a batch to run (recommended to avoid API limits/timeouts)
BATCH_START = 0
BATCH_SIZE = 200
batch_df = Water_Quality_df.iloc[BATCH_START:BATCH_START+BATCH_SIZE].copy()
batch_df.shape

### Note

The Landsat data extraction process for all 9,319 locations typically requires more than 7 hours when executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors, or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.

In this notebook, we provide a sample code snippet demonstrating how to extract data for the first 200 locations. Participants are encouraged to follow the same batching approach to extract data for all 9,319 locations safely and efficiently.

We have already executed the full extraction for all 9,319 locations and saved the output to **landsat_features_training.csv**, which will be used in the benchmark notebook.  
Similarly, participants can extract Landsat features in batches, combine the batch outputs, and save the final merged dataset as **landsat_features_training.csv** to ensure the benchmark notebook runs smoothly.


In [ ]:
# Extract band values + indices from Landsat for the selected TRAINING batch
out_name = f'landsat_features_training_{BATCH_START}_{BATCH_START+BATCH_SIZE-1}.csv'
out_path = f'/tmp/{out_name}'

print('Running Landsat feature extraction for TRAINING batch...')
landsat_train_features = batch_df.progress_apply(compute_Landsat_values, axis=1)

# Attach join keys (kept for merging; do NOT use Lat/Lon as model features)
landsat_train_features['Latitude'] = batch_df['Latitude'].values
landsat_train_features['Longitude'] = batch_df['Longitude'].values
landsat_train_features['Sample Date'] = batch_df['Sample Date'].values

# Column order
cols = ['Latitude','Longitude','Sample Date','blue','green','red','nir','swir16','swir22','NDVI','NDMI','MNDWI','NDWI','NBR']
landsat_train_features = landsat_train_features.reindex(columns=cols)

landsat_train_features.to_csv(out_path, index=False)
print('Wrote:', out_path)


**NDMI and MNDWI Indices**

In this notebook, we compute two commonly used water-related indices from the extracted Landsat bands:

- **NDMI (Normalized Difference Moisture Index):**  
  Measures vegetation water content and surface moisture.  
  Computed as *(NIR - SWIR16) / (NIR + SWIR16)*.

- **MNDWI (Modified Normalized Difference Water Index):**  
  Highlights open water features by enhancing water reflectance and suppressing built-up areas.  
  Computed as *(Green - SWIR16) / (Green + SWIR16)*.

An **epsilon value** (*eps = 1e-10*) is added to the denominators to avoid division by zero.  
These indices are widely used in hydrological and water quality analyses for detecting water presence and vegetation moisture levels.


In [ ]:
# Upload the batch CSV to the notebook stage (optional)
session.sql(f"""
PUT file://{out_path}
snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
AUTO_COMPRESS=FALSE
OVERWRITE=TRUE
""").collect()
print('File saved! Refresh the browser to see the files in the sidebar')


### Extracting features for the validation dataset

In [11]:
Validation_df = read_csv_strict('submission_template.csv')
display(Validation_df.head())

In [12]:
Validation_df.shape

In [ ]:
# Extract band values + indices from Landsat for VALIDATION (submission) dataset
val_out_name = 'landsat_features_validation.csv'
val_out_path = f'/tmp/{val_out_name}'

print('Running Landsat feature extraction for VALIDATION dataset...')
landsat_val_features = Validation_df.progress_apply(compute_Landsat_values, axis=1)

landsat_val_features['Latitude'] = Validation_df['Latitude'].values
landsat_val_features['Longitude'] = Validation_df['Longitude'].values
landsat_val_features['Sample Date'] = Validation_df['Sample Date'].values

cols = ['Latitude','Longitude','Sample Date','blue','green','red','nir','swir16','swir22','NDVI','NDMI','MNDWI','NDWI','NBR']
landsat_val_features = landsat_val_features.reindex(columns=cols)

landsat_val_features.to_csv(val_out_path, index=False)
print('Wrote:', val_out_path)


In [ ]:
# Upload the validation CSV to the notebook stage (optional)
session.sql(f"""
PUT file://{val_out_path}
snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/
AUTO_COMPRESS=FALSE
OVERWRITE=TRUE
""").collect()
print('File saved! Refresh the browser to see the files in the sidebar')
